## Подготовка модели

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
model_name = "Qwen/Qwen2.5-3B-Instruct"

In [3]:
!pip install -q --upgrade torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 83.6 MB/s eta 0:00:00


In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import torch

adapter_path_one = "/content/drive/MyDrive/qwen-russian-lora/checkpoint-154"
adapter_path_two = "/content/drive/MyDrive/qwen-russian-lora-2.0/checkpoint-616"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

custom_model_one = PeftModel.from_pretrained(base_model, adapter_path_one)
custom_model_two = PeftModel.from_pretrained(base_model, adapter_path_two)

custom_model_one.eval()
custom_model_two.eval()


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 2048)
        (layers): ModuleList(
          (0-35): 36 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=2048, out_features=2048, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=32, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=32, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear(

In [6]:
def call_llm(in_model, in_tokenizer, history):

  input_text = tokenizer.apply_chat_template(
      history,
      tokenize=False,
      add_generation_prompt=True
  )

  inputs = in_tokenizer(
      input_text,
      return_tensors="pt"
  ).to(in_model.device)


  with torch.no_grad():
      output = in_model.generate(
          **inputs,
          max_new_tokens=120,
          temperature=1,
          do_sample=True
      )
  #отрезаем вход
  generated_tokens = output[0][inputs["input_ids"].shape[-1]:]

  response = tokenizer.decode(generated_tokens, skip_special_tokens=True)
  return response

##Подготовка аудио

In [9]:
#@title Install dependencies

!pip install -q omegaconf

import torch
from pprint import pprint
from omegaconf import OmegaConf
from IPython.display import Audio, display

torch.hub.download_url_to_file('https://raw.githubusercontent.com/snakers4/silero-models/master/models.yml',
                               'latest_silero_models.yml',
                               progress=False)
models = OmegaConf.load('latest_silero_models.yml')

In [10]:
import torch

language = 'ru'
model_id = 'v5_5_ru'
device = torch.device('cuda')

model, example_text = torch.hub.load(repo_or_dir='snakers4/silero-models',
                                     model='silero_tts',
                                     language=language,
                                     speaker=model_id)
model.to(device)  # gpu or cpu

The repository snakers4_silero-models does not belong to the list of trusted repositories and as such cannot be downloaded. Do you trust this repository and wish to add it to the trusted list of repositories (y/N)?y
Downloading: "https://github.com/snakers4/silero-models/zipball/master" to /root/.cache/torch/hub/master.zip


100%|██████████| 139M/139M [00:02<00:00, 70.0MB/s]
<torch_package_0>.multi_acc_v3_package.py:286: SyntaxWarning: invalid escape sequence '\^'
  text = re.sub(r'[^{}]'.format(self.symbols[3:] + '\^'), '', text)


ниже пример использования

In [11]:
sample_rate = 48000
speaker = 'xenia'
put_accent=True
put_yo=True
put_stress_homo=True
put_yo_homo=True

example_text = 'Меня зовут Лева Королев. Я из готов. И я уже готов открыть все ваши замки любой сложности!'

audio = model.apply_tts(text=example_text,
                        speaker=speaker,
                        sample_rate=sample_rate,
                        put_accent=put_accent,
                        put_yo=put_yo,
                        put_stress_homo=put_stress_homo,
                        put_yo_homo=put_yo_homo)
print(example_text)
display(Audio(audio, rate=sample_rate))

Меня зовут Лева Королев. Я из готов. И я уже готов открыть все ваши замки любой сложности!


## Основная часть

In [12]:
import ipywidgets as widgets
from IPython.display import display

history = []
out = widgets.Output()
inp = widgets.Textarea(placeholder="Введите сообщение...")
btn = widgets.Button(description="Отправить")

def on_send(b):
    text = inp.value.strip()
    if not text: return
    history.append({"role": "user", "content": text})
    inp.value = ""
    with out:
        print(f"Вы: {text}")
    answer = call_llm(custom_model_one, tokenizer, history)
    history.append({"role": "assistant", "content": answer})
    with out:
        print(f"Бот: {answer}\n")

btn.on_click(on_send)
display(out, inp, btn)

Output()

Textarea(value='', placeholder='Введите сообщение...')

Button(description='Отправить', style=ButtonStyle())

In [16]:
def get_audio_from_history(history):
  for block in history:
    audio = model.apply_tts(text=block.get('content'),
                        speaker=speaker,
                        sample_rate=sample_rate,
                        put_accent=put_accent,
                        put_yo=put_yo,
                        put_stress_homo=put_stress_homo,
                        put_yo_homo=put_yo_homo)
    display(Audio(audio, rate=sample_rate))

In [17]:
get_audio_from_history(history)